# 🔐 CyberNews Scraper — Google Colab

Pipeline para recopilar, puntuar y distribuir las **4 noticias más relevantes de ciberseguridad** del mes.

### Fuentes
The Hacker News · SecurityWeek · Kaspersky Securelist · WeLiveSecurity ES

### Pasos
1. **Paso 1** – Cargar el código desde GitHub
2. **Paso 2** – Instalar dependencias
3. **Paso 3** – Ajustar parámetros de ejecución
4. **Paso 4** – Ejecutar el pipeline
5. **Paso 5** – Ver resultados en el notebook
6. **Paso 6** – Descargar archivos _(opcional)_

> El envío por email / Teams / Slack se configura en una fase posterior.


In [ ]:
#@title 🔧 Paso 1 – Clonar el proyecto desde GitHub

import os
import sys
import shutil
import subprocess
from pathlib import Path

PROJECT_DIR = Path('/content/cybernews-scraper')

# Volver a un directorio seguro ANTES de borrar (evita "No such file or directory")
os.chdir('/content')

if PROJECT_DIR.exists():
    shutil.rmtree(str(PROJECT_DIR))

result = subprocess.run(
    ['git', 'clone', 'https://github.com/dalesos92/cybernews_scraper.git', str(PROJECT_DIR)],
    capture_output=True,
    text=True,
)
print(result.stderr)  # git clone escribe progreso en stderr
if result.returncode != 0:
    raise RuntimeError('Clonado fallido. Revisa el mensaje de arriba.')

os.chdir(str(PROJECT_DIR))
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print(f'✓ Directorio: {os.getcwd()}')
print(f'  Archivos: {sorted(p.name for p in PROJECT_DIR.iterdir() if not p.name.startswith("."))[:12]}')


In [ ]:
#@title 📦 Paso 2 – Instalar dependencias

import subprocess
import os
from pathlib import Path

# Localizar requirements.txt: primero en el cwd establecido por el Paso 1,
# luego en la ruta fija de extracción, y finalmente en el directorio del notebook.
def _find_req():
    candidates = [
        Path(os.getcwd()) / 'requirements.txt',
        Path('/content/cybernews-scraper/requirements.txt'),
    ]
    try:
        candidates.insert(0, Path(PROJECT_DIR) / 'requirements.txt')
    except NameError:
        pass
    for p in candidates:
        if p.exists():
            return p
    return None

_req = _find_req()
if _req is None:
    raise FileNotFoundError(
        'No se encontró requirements.txt. '
        'Asegúrate de haber ejecutado el Paso 1 antes.'
    )

print(f'✓ requirements.txt encontrado en: {_req}')
print('Instalando dependencias...')
result = subprocess.run(
    ['pip', 'install', '-q', '-r', str(_req)],
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print('⚠ Advertencias / errores durante la instalación:')
    print((result.stderr or result.stdout)[-2000:])
else:
    print('✓ Dependencias instaladas correctamente')


In [ ]:
#@title ⚙️ Paso 3 – Parámetros de ejecución

#@markdown #### Contenido
top_n = 4  #@param {type:"integer"}
lookback_days = 35  #@param {type:"integer"}

#@markdown #### Fuentes activas
enable_hackernews     = True   #@param {type:"boolean"}
enable_securityweek   = True   #@param {type:"boolean"}
enable_kaspersky      = True   #@param {type:"boolean"}
enable_welivesecurity = True   #@param {type:"boolean"}
enable_cybersecnews   = False  #@param {type:"boolean"}
enable_kaspersky_latam = True  #@param {type:"boolean"}
enable_cybersecnews_es = True  #@param {type:"boolean"}
enable_hispasec = True  #@param {type:"boolean"}
enable_revista_ciberseguridad = True  #@param {type:"boolean"}

print(f'✓ top_n={top_n} | lookback_days={lookback_days}')
_fuentes = [
    ('HackerNews',      enable_hackernews),
    ('SecurityWeek',    enable_securityweek),
    ('Kaspersky',       enable_kaspersky),
    ('WeLiveSec',       enable_welivesecurity),
    ('CyberSecNews',    enable_cybersecnews),
    ('KasperskyLATAM',  enable_kaspersky_latam),
    ('CyberSecNewsES',  enable_cybersecnews_es),
    ('Hispasec',        enable_hispasec),
    ('RevistaCiber',    enable_revista_ciberseguridad),
]

print('  Fuentes: ' + ', '.join(f'{n}={"ON" if v else "off"}' for n, v in _fuentes))

In [ ]:
#@title 🚀 Paso 4 – Ejecutar pipeline

import os
from pathlib import Path

# Generar .env solo con las variables necesarias para ver noticias
_env_vars = {
    'LOG_LEVEL':             'INFO',
    'OUTPUT_DIR':            'output',
    'DB_PATH':               'data/cybernews.db',
    'HTTP_TIMEOUT':          '30',
    'HTTP_MAX_RETRIES':      '3',
    'TOP_N':                 str(top_n),
    'LOOKBACK_DAYS':         str(lookback_days),
    'ENABLE_HACKERNEWS':     str(enable_hackernews).lower(),
    'ENABLE_SECURITYWEEK':   str(enable_securityweek).lower(),
    'ENABLE_KASPERSKY':        str(enable_kaspersky).lower(),
    'ENABLE_WELIVESECURITY':   str(enable_welivesecurity).lower(),
    'ENABLE_CYBERSECNEWS':     str(enable_cybersecnews).lower(),
    'ENABLE_KASPERSKY_LATAM':  str(enable_kaspersky_latam).lower(),
    'ENABLE_CYBERSECNEWS_ES':  str(enable_cybersecnews_es).lower(),
    'ENABLE_HISPASEC':          str(enable_hispasec).lower(),
    'ENABLE_REVISTA_CIBERSEGURIDAD': str(enable_revista_ciberseguridad).lower(),
}
Path('.env').write_text('\n'.join(f'{k}={v}' for k, v in _env_vars.items()), encoding='utf-8')
print('✓ .env generado')
print('=' * 60)

# --dry-run: genera archivos de salida pero no guarda en BD ni envía nada
os.system('python main.py --dry-run')


In [ ]:
#@title 📊 Paso 5 – Ver resultados

import json
from pathlib import Path
from IPython.display import HTML, Markdown, display

_out = Path('output')

# Resumen en texto
_json_path = _out / 'top4_monthly.json'
if _json_path.exists():
    _data = json.loads(_json_path.read_text(encoding='utf-8'))
    print(f"Generado : {_data['generated_at']}")
    print(f"Asunto   : {_data['subject']}")
    print()
    for item in _data['items']:
        print(f"  #{item['rank']} [{item['source']}]")
        print(f"     {item['title']}")
        print(f"     {item['url']}")
        print()
else:
    print('⚠ top4_monthly.json no encontrado. ¿Ejecutaste el Paso 4?')

# Markdown renderizado
_md_path = _out / 'top4_monthly.md'
if _md_path.exists():
    print('\n' + '─' * 50)
    display(Markdown(_md_path.read_text(encoding='utf-8')))

# Preview HTML del email
_html_path = _out / 'top4_email.html'
if _html_path.exists():
    print('\n' + '─' * 50)
    display(HTML(_html_path.read_text(encoding='utf-8')))


In [ ]:
#@title 💾 Paso 6 – Descargar archivos de salida (opcional)

from pathlib import Path

try:
    from google.colab import files as colab_files
    _found = 0
    for _fname in ['top4_monthly.json', 'top4_monthly.md', 'top4_email.html']:
        _fpath = Path('output') / _fname
        if _fpath.exists():
            colab_files.download(str(_fpath))
            print(f'✓ Descargando {_fname}')
            _found += 1
    if _found == 0:
        print('⚠ No se encontraron archivos. Ejecuta el Paso 4 primero.')
except ImportError:
    print('ℹ Esta celda solo funciona en Google Colab.')
